# Vision Transformer

In [10]:
import matplotlib.pyplot as plt
import torch 
import torchvision

from torch import nn
from torchvision import transforms, datasets
from torchinfo import summary
from torch.utils.data import DataLoader,Subset

In [4]:
if torch.cuda.is_available():
    device = "cuda" # Use NVIDIA GPU 
elif torch.backends.mps.is_available():
    device = "mps" # Use Apple Silicon GPU 
else:
    device = "cpu" # Default to CPU if no GPU is available



In [5]:
device

'cuda'

In [6]:
IMG_SIZE = 224
seed = 42

In [7]:
def to_rgb(img):
    """Convert any PIL image to 3-channel RGB (some Caltech101 images are grayscale)."""
    return img.convert("RGB")

In [8]:
# --- TRANSFORMS ---

train_tf = transforms.Compose([
    transforms.Lambda(to_rgb),  # grayscale -> RGB
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15), ratio=(0.3, 3.3), value="random"),
])

test_tf = transforms.Compose([
    transforms.Lambda(to_rgb),
    transforms.Resize(256),            # aspect ratio korunsun
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])


In [11]:
from torch.utils.data import random_split
root = "./data"
test_ratio = 0.2

# --- DATASETS (2 ayrı instance) ---
ds_train_full = datasets.Caltech101(root=root, download=True, transform=train_tf)
ds_test_full  = datasets.Caltech101(root=root, download=True, transform=test_tf)

# --- SPLIT için index üret (reproducible) ---
n_total = len(ds_train_full)
n_test = int(n_total * test_ratio)
n_train = n_total - n_test

g = torch.Generator().manual_seed(seed)
perm = torch.randperm(n_total, generator=g).tolist()

train_idx = perm[:n_train]
test_idx  = perm[n_train:]

# --- SUBSET oluştur (aynı index, farklı transform) ---
train_ds = Subset(ds_train_full, train_idx)
test_ds  = Subset(ds_test_full,  test_idx)

In [12]:
BATCH_SIZE = 32
train_dataloader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader   = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [13]:
print(ds_train_full.categories)

['Faces', 'Faces_easy', 'Leopards', 'Motorbikes', 'accordion', 'airplanes', 'anchor', 'ant', 'barrel', 'bass', 'beaver', 'binocular', 'bonsai', 'brain', 'brontosaurus', 'buddha', 'butterfly', 'camera', 'cannon', 'car_side', 'ceiling_fan', 'cellphone', 'chair', 'chandelier', 'cougar_body', 'cougar_face', 'crab', 'crayfish', 'crocodile', 'crocodile_head', 'cup', 'dalmatian', 'dollar_bill', 'dolphin', 'dragonfly', 'electric_guitar', 'elephant', 'emu', 'euphonium', 'ewer', 'ferry', 'flamingo', 'flamingo_head', 'garfield', 'gerenuk', 'gramophone', 'grand_piano', 'hawksbill', 'headphone', 'hedgehog', 'helicopter', 'ibis', 'inline_skate', 'joshua_tree', 'kangaroo', 'ketch', 'lamp', 'laptop', 'llama', 'lobster', 'lotus', 'mandolin', 'mayfly', 'menorah', 'metronome', 'minaret', 'nautilus', 'octopus', 'okapi', 'pagoda', 'panda', 'pigeon', 'pizza', 'platypus', 'pyramid', 'revolver', 'rhino', 'rooster', 'saxophone', 'schooner', 'scissors', 'scorpion', 'sea_horse', 'snoopy', 'soccer_ball', 'stapler

In [14]:
print("Number of classes:", len(ds_train_full.categories))

Number of classes: 101


In [17]:
height = 224 # H ("The training resolution is 224.")
width = 224 # W
color_channels = 3 # C
patch_size = 16 # P

# Calculate N (number of patches)
number_of_patches = int((height * width) / patch_size**2)
print(f"Number of patches (N) with image height (H={height}), width (W={width}) and patch size (P={patch_size}): {number_of_patches}")

Number of patches (N) with image height (H=224), width (W=224) and patch size (P=16): 196


In [18]:
# Input shape (this is the size of a single image)
embedding_layer_input_shape = (height, width, color_channels)

# Output shape
embedding_layer_output_shape = (number_of_patches, patch_size**2 * color_channels)

print(f"Input shape (single 2D image): {embedding_layer_input_shape}")
print(f"Output shape (single 2D image flattened into patches): {embedding_layer_output_shape}")

Input shape (single 2D image): (224, 224, 3)
Output shape (single 2D image flattened into patches): (196, 768)
